# Chương 7: Xây dựng Ứng dụng Chat
## Bắt đầu nhanh với OpenAI API

Sổ tay này được điều chỉnh từ [Kho Mẫu Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) bao gồm các sổ tay truy cập dịch vụ [Azure OpenAI](notebook-azure-openai.ipynb).

API Python của OpenAI cũng hoạt động với các Mô hình Azure OpenAI, chỉ với một vài chỉnh sửa. Tìm hiểu thêm về sự khác biệt tại đây: [Cách chuyển đổi giữa các điểm cuối OpenAI và Azure OpenAI với Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Tổng quan  
"Các mô hình ngôn ngữ lớn là các hàm ánh xạ văn bản sang văn bản. Được cung cấp một chuỗi văn bản đầu vào, một mô hình ngôn ngữ lớn cố gắng dự đoán văn bản sẽ tiếp theo"(1). Sổ tay "khởi đầu nhanh" này sẽ giới thiệu cho người dùng các khái niệm LLM ở cấp độ cao, các yêu cầu gói cốt lõi để bắt đầu với AML, một sự giới thiệu nhẹ nhàng về thiết kế lời nhắc (prompt), và một số ví dụ ngắn về các trường hợp sử dụng khác nhau. 


## Mục lục  

[Tổng quan](#overview)  
[Cách sử dụng Dịch vụ OpenAI](#how-to-use-openai-service)  
[1. Tạo Dịch vụ OpenAI của bạn](#1.-creating-your-openai-service)  
[2. Cài đặt](#2.-installation)    
[3. Thông tin xác thực](#3.-credentials)  

[Các trường hợp sử dụng](#use-cases)    
[1. Tóm tắt văn bản](#1.-summarize-text)  
[2. Phân loại văn bản](#2.-classify-text)  
[3. Tạo tên sản phẩm mới](#3.-generate-new-product-names)  
[4. Tinh chỉnh bộ phân loại](#4.fine-tune-a-classifier)  

[Tài liệu tham khảo](#references)


### Xây dựng lời nhắc đầu tiên của bạn  
Bài tập ngắn này sẽ cung cấp một giới thiệu cơ bản về cách gửi lời nhắc đến một mô hình OpenAI cho một nhiệm vụ đơn giản "tóm tắt".


**Các bước**:  
1. Cài đặt thư viện OpenAI trong môi trường python của bạn  
2. Tải các thư viện trợ giúp tiêu chuẩn và đặt các thông tin bảo mật OpenAI tiêu chuẩn cho Dịch vụ OpenAI mà bạn đã tạo  
3. Chọn một mô hình cho nhiệm vụ của bạn  
4. Tạo một lời nhắc đơn giản cho mô hình  
5. Gửi yêu cầu của bạn đến API mô hình!


### 1. Cài đặt OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Nhập các thư viện hỗ trợ và khởi tạo thông tin đăng nhập


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Tìm mô hình phù hợp  
Các mô hình như GPT-4o và GPT-4o mini có thể hiểu và tạo ra ngôn ngữ tự nhiên, và có sẵn trên nền tảng OpenAI với các cấp độ sức mạnh và tốc độ khác nhau phù hợp với các nhiệm vụ khác nhau.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Thiết kế câu lệnh  

"Phép màu của các mô hình ngôn ngữ lớn là bằng cách được huấn luyện để giảm thiểu sai số dự đoán này trên một lượng lớn văn bản, các mô hình cuối cùng học được các khái niệm hữu ích cho những dự đoán này. Ví dụ, chúng học được các khái niệm như"(1):

* cách đánh vần
* cách ngữ pháp hoạt động
* cách diễn giải lại
* cách trả lời câu hỏi
* cách giữ cuộc hội thoại
* cách viết bằng nhiều thứ tiếng
* cách lập trình
* v.v.

#### Cách điều khiển một mô hình ngôn ngữ lớn  
"Trong tất cả các đầu vào cho một mô hình ngôn ngữ lớn, rõ ràng đầu vào có ảnh hưởng nhất là câu lệnh văn bản(1).

Các mô hình ngôn ngữ lớn có thể được điều khiển phát sinh kết quả bằng một vài cách:

Hướng dẫn: Nói với mô hình điều bạn muốn
Hoàn thành: Khiến mô hình hoàn thành phần bắt đầu những gì bạn muốn
Minh họa: Cho mô hình thấy điều bạn muốn, với:
Một vài ví dụ trong câu lệnh
Nhiều trăm hoặc nghìn ví dụ trong bộ dữ liệu huấn luyện tinh chỉnh"



#### Có ba nguyên tắc cơ bản để tạo câu lệnh:

**Cho thấy và nói rõ**. Làm cho rõ ràng điều bạn muốn thông qua hướng dẫn, ví dụ, hoặc kết hợp cả hai. Nếu bạn muốn mô hình xếp hạng một danh sách các mục theo thứ tự chữ cái hoặc phân loại một đoạn văn theo cảm xúc, hãy cho nó thấy đó là điều bạn muốn.

**Cung cấp dữ liệu chất lượng**. Nếu bạn đang cố gắng xây dựng bộ phân loại hoặc khiến mô hình tuân theo một khuôn mẫu, hãy chắc chắn rằng có đủ ví dụ. Đảm bảo đọc lại các ví dụ của bạn — mô hình thường đủ thông minh để nhìn thấy các lỗi chính tả cơ bản và đưa ra phản hồi, nhưng nó cũng có thể cho rằng đó là cố ý và điều đó có thể ảnh hưởng đến phản hồi.

**Kiểm tra cài đặt của bạn.** Các cài đặt temperature và top_p kiểm soát mô hình có tính xác định trong việc tạo phản hồi. Nếu bạn đang yêu cầu nó một phản hồi chỉ có một câu trả lời đúng duy nhất, thì bạn nên đặt các cài đặt này thấp hơn. Nếu bạn muốn phản hồi đa dạng hơn, thì bạn có thể đặt cao hơn. Sai lầm số một mà mọi người mắc phải với các cài đặt này là cho rằng chúng là điều khiển "thông minh" hoặc "sáng tạo".


Nguồn: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Nộp bài!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Gọi lại lệnh đó một lần nữa, kết quả sẽ so sánh thế nào?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Tóm Tắt Văn Bản  
#### Thách Thức  
Tóm tắt văn bản bằng cách thêm 'tl;dr:' vào cuối đoạn văn bản. Hãy chú ý cách mô hình hiểu để thực hiện nhiều nhiệm vụ mà không cần hướng dẫn thêm. Bạn có thể thử nghiệm với các lệnh mô tả cụ thể hơn thay vì tl;dr để thay đổi hành vi của mô hình và tùy chỉnh bản tóm tắt bạn nhận được(3).  

Các công trình gần đây đã chứng minh sự cải thiện đáng kể trên nhiều nhiệm vụ và bộ đánh giá NLP bằng cách tiền huấn luyện trên một kho dữ liệu văn bản lớn, sau đó tinh chỉnh thêm cho một nhiệm vụ cụ thể. Mặc dù thường không phụ thuộc vào nhiệm vụ trong kiến trúc, phương pháp này vẫn yêu cầu các bộ dữ liệu tinh chỉnh đặc thù cho nhiệm vụ gồm hàng ngàn hoặc hàng chục ngàn ví dụ. Ngược lại, con người thường có thể thực hiện một nhiệm vụ ngôn ngữ mới chỉ với vài ví dụ hoặc hướng dẫn đơn giản - điều mà các hệ thống NLP hiện nay vẫn còn gặp nhiều khó khăn. Ở đây chúng tôi cho thấy việc mở rộng các mô hình ngôn ngữ cải thiện đáng kể hiệu năng không phụ thuộc nhiệm vụ với vài ví dụ, đôi khi còn đạt được sự cạnh tranh với các phương pháp tinh chỉnh tiên tiến trước đây.  



Tl;dr


# Bài tập cho một số trường hợp sử dụng  
1. Tóm tắt văn bản  
2. Phân loại văn bản  
3. Tạo tên sản phẩm mới


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Phân loại văn bản  
#### Thách thức  
Phân loại các mục vào các danh mục được cung cấp tại thời điểm suy luận. Trong ví dụ sau, chúng tôi cung cấp cả các danh mục và văn bản cần phân loại trong lời nhắc (*playground_reference). 

Yêu cầu của khách hàng: Xin chào, một trong những phím trên bàn phím laptop của tôi gần đây đã bị hỏng và tôi cần thay thế:

Danh mục được phân loại:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Tạo Tên Sản Phẩm Mới
#### Thách Thức
Tạo tên sản phẩm từ các từ ví dụ. Ở đây chúng tôi bao gồm trong lời nhắc thông tin về sản phẩm mà chúng tôi sẽ tạo tên cho. Chúng tôi cũng cung cấp một ví dụ tương tự để thể hiện mẫu mà chúng tôi mong muốn nhận được. Chúng tôi cũng đã đặt giá trị nhiệt độ cao để tăng tính ngẫu nhiên và các phản hồi sáng tạo hơn.

Mô tả sản phẩm: Máy làm sữa lắc tại nhà
Từ khóa gợi ý: nhanh, lành mạnh, nhỏ gọn.
Tên sản phẩm: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Mô tả sản phẩm: Một đôi giày có thể vừa với mọi kích cỡ chân.
Từ khóa gợi ý: thích ứng, vừa vặn, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Tài liệu tham khảo  
- [Sổ tay Openai](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Cổng thông tin Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Các thực hành tốt nhất để điều chỉnh tinh vi các mô hình GPT nhằm phân loại văn bản](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Để được hỗ trợ thêm  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Những người đóng góp
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
